# Partie 3 - Apprentissage

On veut prédire le **pourcentage de victoires** d'un Pokémon. C'est une **valeur numérique**, donc on utilise des algorithmes de **régression** (apprentissage supervisé) :
- régression linéaire;
- arbre de décision;
- forêt aléatoire.

## Chargement des données
On charge le `dataset.csv` créé à la partie 2, puis on supprime les lignes qui ont des valeurs manquantes.

In [7]:
import pandas as pnd

# Chargement du dataset avec pour séparateur des tabulations
dataset = pnd.read_csv("datas/dataset.csv", delimiter='\t')

# Suppression des lignes ayant des valeurs manquantes
dataset = dataset.dropna(axis=0, how='any')

print(dataset.shape)

(404, 18)


## Choix des données d'apprentissage

### Variables prédictives (X)
Ce sont les features à partir desquelles la machine apprend (colonnes 5 à 11) :
- `POINTS_DE_VIE`
- `NIVEAU_ATTAQUE` et `NIVEAU_ATTAQUE_SPECIALE`
- `NIVEAU_DEFENSE` et `NIVEAU_DEFENSE_SPECIALE`
- `VITESSE`
- `NOMBRE_GENERATIONS`

`iloc[:, 5:12]` veut dire : **toutes les lignes** (`:`), colonnes **5 à 11** (le 12 est exclu).

In [8]:
#X = on prend toutes les données, mais uniquement les features 5 à 12 
#(ne pas hésiter à ouvrir le fichier afin de bien visualiser les features concernées)
#POINTS_ATTAQUE;POINTS_DEFFENCE;POINTS_ATTAQUE_SPECIALE;POINT_DEFENSE_SPECIALE;POINTS_VITESSE;NOMBRE_GENERATIONS
X = dataset.iloc[:, 5:12].values

### Variable à prédire (y)
C'est la valeur que la machine doit apprendre à prédire : `POURCENTAGE_DE_VICTOIRES` (colonne 17).

In [9]:
#y = on prend uniquement la feature POURCENTAGE_DE_VICTOIRE (17ème feature)
y = dataset.iloc[:, 17].values

En résumé, la machine va apprendre à prédire le **pourcentage de victoires** d'un Pokémon à partir de ses points de vie, de ses niveaux d'attaque et de défense (standards et spéciaux), de sa vitesse et de son nombre de générations.

X et y seront ensuite découpés en deux groupes :

| Groupe | Contenu | Rôle |
|---|---|---|
| `X_APPRENTISSAGE`, `Y_APPRENTISSAGE` | 80 % des observations | La machine **apprend** |
| `X_VALIDATION`, `Y_VALIDATION` | 20 % des observations | On **vérifie** son apprentissage |

In [14]:
#X = on prend toutes les données, mais uniquement les features 4 à 11
# POINTS_ATTAQUE;POINTS_DEFFENCE;POINTS_ATTAQUE_SPECIALE;POINT_DEFENSE_SPECIALE;POINTS_VITESSE;NOMBRE_GENERATIONS
X = dataset.iloc[:, 5:12].values

#y = on prend uniquement la colonne POURCENTAGE_DE_VICTOIRE (16ème feature) 
#les : signifiant "Pour toutes les observations"
y = dataset.iloc[:, 17].values

#Construction du jeu d'entrainement et du jeu de tests
from sklearn.model_selection import train_test_split
X_APPRENTISSAGE, X_VALIDATION, Y_APPRENTISSAGE, Y_VALIDATION = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Algorithme 1 : Régression linéaire
La régression linéaire modélise la relation entre des **variables prédictives** et une **variable cible**.

- **Variables prédictives** : points de vie, attaque, défense, attaque spéciale, défense spéciale, vitesse et génération.
- **Variable cible** : le pourcentage de victoires.

Comme on utilise plusieurs variables prédictives, c'est une **régression linéaire multiple**.

Étapes :
1. `fit()` : apprentissage sur le jeu d'**apprentissage**;
2. `predict()` : prédictions sur le jeu de **validation**, des données que le modèle n'a jamais vues;
3. `r2_score()` : précision, en comparant les prédictions aux vraies valeurs. C'est la mesure recommandée par [Scikit-Learn](https://scikit-learn.org/stable/modules/model_evaluation.html) pour la régression (1 = parfait).

In [15]:
#---- ALGORITHME 1: REGRESSION LINEAIRE -----
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

#Choix de l'algorithme
algorithme = LinearRegression()

#Apprentissage à l'aide de la fonction fit
#Apprentisage effectué sur le jeu de données d'apprentissage
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)

#Realisation de prédictions sur le jeu de tests (validation)
predictions = algorithme.predict(X_VALIDATION)

#Calcul de la précision de l'apprentissage à l'aide de la
#fonction r2_score en comparant les valeurs prédites
#(predictions) et les valeurs attendues (Y_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- REGRESSION LINEAIRE -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- REGRESSION LINEAIRE -----------
>> Precision = 0.9043488485570964
------------------------------------------


La régression linéaire obtient une précision d'environ **90 %**, ce qui est plutôt bon.

## Algorithme 2 : Arbre de décision
On utilise `DecisionTreeRegressor`, l'arbre de décision pour les problèmes de **régression** (prédire une valeur), et non `DecisionTreeClassifier`, qui sert à la **classification** (prédire une catégorie).

On suit les mêmes étapes : `fit()`, `predict()` et `r2_score()`.

In [16]:
#Choix de l'algorithme
from sklearn.tree import DecisionTreeRegressor
algorithme = DecisionTreeRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- ARBRES DE DECISIONS -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- ARBRES DE DECISIONS -----------
>> Precision = 0.8896081511316302
------------------------------------------


L'arbre de décision obtient une précision d'environ **88 %**, un peu **moins bonne** que la régression linéaire.

## Algorithme 3 : Forêt aléatoire
Une forêt aléatoire combine **plusieurs arbres de décision** et fait la moyenne de leurs prédictions. Elle est généralement plus précise qu'un seul arbre.

On utilise `RandomForestRegressor`, la version pour la **régression**.

In [17]:
#Choix de l'algorithme
from sklearn.ensemble import RandomForestRegressor
algorithme = RandomForestRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- FORETS ALEATOIRES -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- FORETS ALEATOIRES -----------
>> Precision = 0.9359862024540008
------------------------------------------


La forêt aléatoire obtient une précision d'environ **93 %**, la meilleure des trois.

| Algorithme | Précision |
|---|---|
| Régression linéaire | ≈ 90 % |
| Arbre de décision | ≈ 88 % |
| **Forêt aléatoire** | **≈ 93 %** |

C'est donc le modèle qu'on conserve. Les trois algorithmes ont été utilisés **sans optimisation** de leurs paramètres : en les ajustant, on pourrait obtenir des résultats différents, peut-être meilleurs.

# Sauvegarde du modèle d’apprentissage

In [20]:
import os
import joblib

# Création du dossier modele s'il n'existe pas
os.makedirs('modele', exist_ok=True)

# Sauvegarde du modèle
fichier = 'modele/modele_pokemon.mod'
joblib.dump(algorithme, fichier)

['modele/modele_pokemon.mod']

## Vérification du surapprentissage
On compare la précision sur le jeu d'**apprentissage** avec celle sur le jeu de **validation**. Si la précision d'apprentissage est beaucoup plus élevée, le modèle a appris les données par cœur au lieu de généraliser : c'est le **surapprentissage**.

In [22]:
predictions = algorithme.predict(X_VALIDATION)
precision_apprentissage = algorithme.score(X_APPRENTISSAGE,Y_APPRENTISSAGE)
precision = r2_score(Y_VALIDATION, predictions)

print("Précision apprentissage :", precision_apprentissage)
print("Précision validation    :", precision)

Précision apprentissage : 0.991857837634242
Précision validation    : 0.9359862024540008


- **Précision apprentissage** : ≈ **99,2 %**
- **Précision validation** : ≈ **93,6 %**

L'écart est d'environ **5,6 points**. La précision d'apprentissage est plus élevée, ce qui est normal pour une forêt aléatoire, puisqu'elle connaît déjà ces données. Mais elle reste très bonne sur des données jamais vues : il n'y a **pas de surapprentissage important** et le modèle **généralise bien**.